In [1]:
!pip install -q transformers datasets accelerate scikit-learn
import torch
print("GPU available:", torch.cuda.is_available())

GPU available: True


In [2]:
import numpy as np
from datasets import load_dataset, Sequence, Value
from transformers import AutoTokenizer

ds = load_dataset("google-research-datasets/go_emotions", "simplified")
print(ds["train"][0])

model_name = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

NUM_LABELS = 28

def preprocess(batch):
    enc = tokenizer(batch["text"], truncation=True,
                    padding="max_length", max_length=128)
    multi_hot = np.zeros((len(batch["labels"]), NUM_LABELS), dtype=np.float32)
    for i, label_list in enumerate(batch["labels"]):
        multi_hot[i, label_list] = 1.0
    enc["labels"] = multi_hot.tolist()
    return enc

tokenized = ds.map(
    preprocess,
    batched=True,
    remove_columns=ds["train"].column_names,
    load_from_cache_file=False,
)

tokenized = tokenized.cast_column("labels", Sequence(Value("float32"), length=NUM_LABELS))

print(tokenized)
print("Label dtype check:", type(tokenized["train"][0]["labels"][0]))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

...

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/43410 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/5426 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 5427
    })
})
Label dtype check: <class 'float'>


In [3]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="goemotions-distilroberta",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.104785,0.098758
2,0.092089,0.091883


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2714, training_loss=0.11639760484041847, metrics={'train_runtime': 950.2164, 'train_samples_per_second': 91.369, 'train_steps_per_second': 2.856, 'total_flos': 2876538042961920.0, 'train_loss': 0.11639760484041847, 'epoch': 2.0})

In [5]:
import numpy as np
from sklearn.metrics import f1_score, classification_report

# Predict on the test split
preds_output = trainer.predict(tokenized["test"])
logits = preds_output.predictions
labels = preds_output.label_ids

# Sigmoid -> probabilities -> threshold at 0.3
probs = 1 / (1 + np.exp(-logits))
preds = (probs >= 0.3).astype(int)

emotion_names = ds["train"].features["labels"].feature.names

print("Macro F1:", round(f1_score(labels, preds, average="macro"), 3))
print("Micro F1:", round(f1_score(labels, preds, average="micro"), 3))
print()
print(classification_report(labels, preds, target_names=emotion_names, zero_division=0))

Macro F1: 0.375
Micro F1: 0.592

                precision    recall  f1-score   support

    admiration       0.66      0.76      0.70       504
     amusement       0.77      0.89      0.83       264
         anger       0.46      0.44      0.45       198
     annoyance       0.38      0.18      0.25       320
      approval       0.52      0.34      0.41       351
        caring       0.53      0.16      0.24       135
     confusion       0.58      0.31      0.41       153
     curiosity       0.47      0.70      0.56       284
        desire       0.68      0.20      0.31        83
disappointment       0.00      0.00      0.00       151
   disapproval       0.46      0.29      0.36       267
       disgust       0.86      0.15      0.26       123
 embarrassment       0.00      0.00      0.00        37
    excitement       0.87      0.19      0.32       103
          fear       0.89      0.10      0.18        78
     gratitude       0.92      0.92      0.92       352
         grief

In [6]:
trainer.save_model("moodmate-emotion-model")
tokenizer.save_pretrained("moodmate-emotion-model")

# Zip it for download
!zip -r moodmate-emotion-model.zip moodmate-emotion-model

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: moodmate-emotion-model/ (stored 0%)
  adding: moodmate-emotion-model/tokenizer.json (deflated 82%)
  adding: moodmate-emotion-model/tokenizer_config.json (deflated 51%)
  adding: moodmate-emotion-model/config.json (deflated 65%)
  adding: moodmate-emotion-model/training_args.bin (deflated 54%)
  adding: moodmate-emotion-model/model.safetensors (deflated 7%)
